# 손글씨 각장 표기 검출기 추가 학습 (PaddleOCR PP-OCRv6_small_det)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganggang-0605/vision-welding-ai/blob/main/vision/notebooks/colab_train_det.ipynb)

1단계(시각 인식)의 글자 **위치 찾기** 모델을 이어서 학습합니다. 지금 검출기는 운영측 손글씨 예시 1·2에서 글자 위치를 아예 못 찾아서,
인식기(colab_train_rec.ipynb)를 학습해도 실제 사진에서는 읽을 영역이 생기지 않습니다.

**데이터** `hw_det_v3.zip` (vision/tools/build_det_data.py)
- 학습: 합성 손글씨 장면 1,500장(40%는 운영측 예시처럼 표기를 가까이 찍은 작은 사진을 1280px로 키운 것) + steel-ocr 학습용 실제 표기 사진 223장 (기존 표기를 못 찾게 되지 않도록 섞음, 정답 없는 표기는 무시 영역)
- 평가: 합성 150장 · steel-ocr val 96장(기존 표기 유지 확인) · **운영측 사진 5장**(손글씨 예시 3장 + 블록 사진 2장, 상자 21개, 1단계처럼 긴 변 1280px로 키움)
- 운영측 사진이 들어 있으니 공유 범위 주의

**준비** `런타임 → 런타임 유형 변경 → T4 GPU`, `hw_det_v3.zip`을 Drive `내 드라이브/pac_hw/` 또는 Colab `/content/dataset/`에 올리기

**순서** 위에서부터 차례로 실행. 학습은 T4에서 1~1.5시간 예상(Colab CPU 2개라 사진 준비가 느림) — 끊기면 5번만 다시 실행하면 Drive의 체크포인트부터 이어서 학습.
세션을 다시 시작했다면 1 → 2 → (3번 둘째 칸) → 4 → 원하는 칸.

**결과** `pac_hw/runs/PP-OCRv6_small_det_v3/` 의 `inference.zip`(학습한 검출기)과 `eval.json`

## 1. 설정

In [ ]:
LOCAL_DIR = "/content/dataset"                # Colab 파일 창에 올린 곳 (zip 또는 푼 폴더)
DRIVE_DIR = "/content/drive/MyDrive/pac_hw"   # Drive에 올린 곳 + 체크포인트·결과 저장
MODEL = "PP-OCRv6_small_det"                  # 1단계 기본 검출 모델과 같음 (vision/src/vision/ocr.py)
DATA_NAME = "hw_det_v3"                       # 데이터 이름 = zip 이름 = zip 안 폴더 이름
RUN = "v3"                                    # 결과 폴더: runs/PP-OCRv6_small_det_v3
INIT_FROM = None                              # 이어서 학습할 체크포인트 (None: 사전 학습 모델에서 시작)
EPOCHS = 25
LEARNING_RATE = 2e-4                          # 기본값(1e-3)은 GPU 8장 기준 — 1장으로 이어서 학습하니 낮게
BATCH = 8
LIMIT_SIDE = 1280                             # 평가 때 사진 긴 변 — 1단계 검출 설정과 같게 (det_limit_side_len)
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"  # nvidia-smi 의 CUDA 버전이 12.6 미만이면 cu118

SAVE_DIR = f"{DRIVE_DIR}/runs/{MODEL}_{RUN}" if RUN else f"{DRIVE_DIR}/runs/{MODEL}"  # 체크포인트·결과 (Drive)
PRETRAINED = f"/content/pretrained/{MODEL}_pretrained.pdparams"  # 사전 학습 가중치 (3번에서 받음)
CONFIG = "/content/hw_det.yml"                # 학습 설정 (4번에서 만듦)
EXPORT = f"{SAVE_DIR}/inference"              # 학습한 모델 (6번에서 내보냄)
DATA = "/content/data"                        # zip을 푸는 곳 (2번 칸이 데이터 위치로 바꿈)
COMPARE = {"학습 전": PRETRAINED, "이번": f"{SAVE_DIR}/best_accuracy"}  # 7번에서 비교할 모델 (있는 것만)

## 2. GPU 확인, Drive 연결, 데이터 풀기

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print("Python", sys.version.split()[0])

In [ ]:
import glob, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")  # 체크포인트·결과는 Drive에 저장
os.makedirs(SAVE_DIR, exist_ok=True)

def find(pattern):
    """LOCAL_DIR → DATA → DRIVE_DIR 순서로 찾음"""
    for root in [LOCAL_DIR, DATA, DRIVE_DIR]:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return hits[0]

found = find(f"{DATA_NAME}/train.txt")
if found:
    DATA = os.path.dirname(found)
else:
    zip_path = find(f"{DATA_NAME}.zip")
    assert zip_path, f"{DATA_NAME}.zip 이나 푼 폴더({DATA_NAME}/train.txt)를 {LOCAL_DIR} · {DRIVE_DIR} 에서 못 찾음"
    if zip_path.startswith("/content/drive"):  # Drive에서 바로 풀면 느려서 로컬로 복사
        shutil.copy(zip_path, f"/content/{DATA_NAME}.zip")
        zip_path = f"/content/{DATA_NAME}.zip"
    zipfile.ZipFile(zip_path).extractall("/content/data")
    DATA = f"/content/data/{DATA_NAME}"
print("데이터 위치:", DATA)
for name in ["train", "val_synth", "val_steel", "val_pac"]:
    n = sum(1 for line in open(f"{DATA}/{name}.txt", encoding="utf-8") if line.strip())
    print(f"{name}: {n}장")

## 3. 설치 (5~10분)
GPU용 PaddlePaddle 3.3.1(1단계와 같은 버전)과 PaddleOCR v3.7.0 학습 코드.
Colab 기본 PyTorch는 먼저 지움 — PaddlePaddle이 GPU 통신 라이브러리(NCCL)를 바꿔 PyTorch를 불러오는 패키지가 깨짐 (학습·평가에 필요 없음).
설치 뒤 재시작을 요구하면 `런타임 → 세션 다시 시작` 후 1번부터 (설치·가중치는 남아 있어 3번 첫째 칸은 건너뛰어도 됨).

In [ ]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install paddlepaddle-gpu==3.3.1 -i {PADDLE_INDEX}
!test -d /content/PaddleOCR || git clone -q --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!pip -q install -r /content/PaddleOCR/requirements.txt
import paddle
paddle.utils.run_check()

In [ ]:
if not os.path.exists(PRETRAINED):
    os.makedirs("/content/pretrained", exist_ok=True)
    !wget -q -O {PRETRAINED} https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/{MODEL}_pretrained.pdparams
print(PRETRAINED, os.path.getsize(PRETRAINED) // 1_000_000, "MB")

## 4. 학습 설정 만들기
PaddleOCR 기본 설정(`configs/det/PP-OCRv6/PP-OCRv6_small_det.yml`)에서 바꾸는 것:
- 에폭 500 → 25 (사전 학습 모델에서 이어서 학습), 학습률 1e-3(GPU 8장 기준) → 2e-4, 배치 8, Colab용 로딩 프로세스 2개
- 정답 지도(MakeBorderMap · MakeShrinkMap)의 total_epoch 도 에폭 수에 맞춤
- 학습 중 최고 모델은 **합성 + steel-ocr 검증**으로 고름 — 운영측 사진은 고르는 데 쓰지 않고 마지막 평가에만 (평가 세트에 맞추지 않게)
- 평가 때 사진 크기: 긴 변 1280px 이하로 (1단계 검출 설정과 같게)

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"/content/PaddleOCR/configs/det/PP-OCRv6/{MODEL}.yml", encoding="utf-8"))
g = cfg["Global"]
g.update(epoch_num=EPOCHS, save_model_dir=SAVE_DIR, save_epoch_step=10, eval_batch_step=[0, 400], print_batch_step=20,
         pretrained_model=INIT_FROM if INIT_FROM and os.path.exists(INIT_FROM + ".pdparams") else PRETRAINED,
         use_visualdl=False)
cfg["Optimizer"]["lr"].update(learning_rate=LEARNING_RATE, warmup_epoch=1)
train = cfg["Train"]
train["dataset"].update(data_dir=DATA, label_file_list=[f"{DATA}/train.txt"], ratio_list=[1.0])
for t in train["dataset"]["transforms"]:
    for name in ("MakeBorderMap", "MakeShrinkMap"):
        if name in t:
            t[name]["total_epoch"] = EPOCHS
train["loader"].update(batch_size_per_card=BATCH, num_workers=2)
ev = cfg["Eval"]
ev["dataset"].update(data_dir=DATA, label_file_list=[f"{DATA}/val_synth.txt", f"{DATA}/val_steel.txt"])
for i, t in enumerate(ev["dataset"]["transforms"]):
    if "DetResizeForTest" in t:
        ev["dataset"]["transforms"][i] = {"DetResizeForTest": {"limit_side_len": LIMIT_SIDE, "limit_type": "max"}}
ev["loader"]["num_workers"] = 2
yaml.safe_dump(cfg, open(CONFIG, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
shutil.copy(CONFIG, f"{SAVE_DIR}/hw_det.yml")
print("시작 가중치:", g["pretrained_model"])

## 5. 학습 (T4에서 1~1.5시간)
`hmean`(정밀도와 재현율의 조화 평균)이 합성 + steel-ocr 검증 성적. 끊기면 이 칸만 다시 실행 → Drive의 `latest`부터 이어서 학습.
`best metric`의 hmean이 여러 번 평가에서 더 오르지 않으면 멈추고 6번으로 가도 됨 (최고 모델은 `best_accuracy`로 이미 저장됨).

In [ ]:
ckpt = f"{SAVE_DIR}/latest"
resume = f"-o Global.checkpoints={ckpt}" if os.path.exists(ckpt + ".pdparams") else ""
print("이어서 학습" if resume else "처음부터 학습")
!cd /content/PaddleOCR && python -u tools/train.py -c {CONFIG} {resume} 2>&1 | grep --line-buffered -v -i "warn"

## 6. 학습한 모델 내보내기

In [ ]:
!cd /content/PaddleOCR && python tools/export_model.py -c {CONFIG} -o Global.pretrained_model={SAVE_DIR}/best_accuracy Global.save_inference_dir={EXPORT}
!ls -la {EXPORT}
shutil.make_archive(f"{SAVE_DIR}/inference", "zip", EXPORT)
print("저장:", f"{SAVE_DIR}/inference.zip")

## 7. 평가: 학습 전 vs 학습 후
- **운영측 5장** — 실제 과제 사진에서 손글씨 위치를 찾는지 (핵심). 학습 전에는 손글씨 예시 1·2에서 아무것도 못 찾음.
  정답 상자를 사람이 넉넉하게 그려서 IoU 0.5 기준 점수는 낮게 나올 수 있음 — 아래 상자 그림으로 같이 판단
- **steel-ocr val 96장** — 기존 표기 찾기가 떨어지지 않았는지
- **합성 150장** — 학습에 안 쓴 글씨체
- 정밀도(precision): 찾은 상자 중 맞은 비율 · 재현율(recall): 정답 중 찾은 비율 · hmean: 둘의 조화 평균

In [ ]:
import json, re, subprocess

def det_eval(weights, list_name):
    cmd = ["python", "tools/eval.py", "-c", CONFIG, "-o", f"Global.pretrained_model={weights}",
           f"Eval.dataset.label_file_list=['{DATA}/{list_name}.txt']"]
    run = subprocess.run(cmd, cwd="/content/PaddleOCR", capture_output=True, text=True)
    text = run.stdout + run.stderr
    tail = text.split("metric eval")[-1]
    found = dict(re.findall(r"(precision|recall|hmean):\s*([0-9.]+)", tail))
    if run.returncode != 0 or not found:
        print(text[-3000:])
        raise RuntimeError(f"{list_name} 평가 실패")
    return {k: round(float(v), 4) for k, v in found.items()}

models = {name: w for name, w in COMPARE.items() if os.path.exists(w) or os.path.exists(w + ".pdparams")}
report = {name: {lst: det_eval(w, lst) for lst in ["val_pac", "val_steel", "val_synth"]} for name, w in models.items()}
print(f"{'':10s}" + "".join(f"{lst:>28s}" for lst in ["val_pac (운영측)", "val_steel", "val_synth"]))
for name, r in report.items():
    print(f"{name:10s}" + "".join(f"   P {m['precision']:.2f} R {m['recall']:.2f} H {m['hmean']:.2f}" for m in r.values()))
json.dump(report, open(f"{SAVE_DIR}/eval.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("저장:", f"{SAVE_DIR}/eval.json")

### 운영측 사진에서 찾은 상자 보기 (손글씨 예시 3장 + 블록 사진 2장)

In [ ]:
from IPython.display import Image, display
pac_dir = f"{DATA}/pac/images"
for k, (name, w) in enumerate(models.items()):
    out = f"/content/det_out/{k}/predicts.txt"
    subprocess.run(["python", "tools/infer_det.py", "-c", CONFIG, "-o", f"Global.pretrained_model={w}",
                    f"Global.infer_img={pac_dir}", f"Global.save_res_path={out}"], cwd="/content/PaddleOCR",
                   capture_output=True, text=True, check=True)
    print(name)
    for path in sorted(glob.glob(f"/content/det_out/{k}/det_results/pac_*")):
        display(Image(filename=path, width=360))

## 다음
- `inference.zip`(검출기)과 인식기 노트북의 `inference.zip`을 저장소 담당자에게 전달 → 1단계에 두 모델을 연결하고
  운영측 사진 전체 파이프라인과 steel-ocr·MPSC 인쇄체 회귀를 원격으로 확인